# Assessment 2: Knowledge Graph with Semantic Search

In [1]:
!pip install -q rdflib faiss-cpu sentence-transformers

## 1. Building RDF knowledge graph

In [2]:
from rdflib import Graph, Namespace, RDF, Literal

EX = Namespace("http://example.org/")
g = Graph()
g.bind("ex", EX)

g.add((EX.Alice, RDF.type, EX.Person))
g.add((EX.Bob, RDF.type, EX.Person))
g.add((EX.Alice, EX.knows, EX.Bob))
g.add((EX.Bob, EX.worksAt, EX.AcmeCorp))

for s, p, o in g:
    print(s.split("/")[-1], p.split("/")[-1].split("#")[-1], o.split("/")[-1])

Alice type Person
Alice knows Bob
Bob worksAt AcmeCorp
Bob type Person


## 2. Converting triples to sentences

In [3]:
import re

def name(x):
    return re.sub(r"(?<!^)(?=[A-Z])", " ", str(x).split("/")[-1].split("#")[-1])

def to_sentence(s, p, o):
    pred = name(p).lower()
    if pred == "type":
        return f"{name(s)} is a {name(o).lower()}"
    return f"{name(s)} {pred} {name(o)}"

sentences = [to_sentence(s, p, o) for s, p, o in g]
sentences

['Alice is a person',
 'Alice knows Bob',
 'Bob works at Acme Corp',
 'Bob is a person']

## 3. Embed sentences

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
emb = model.encode(sentences, normalize_embeddings=True).astype("float32")
emb.shape

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(4, 384)

## 4. Storing in FAISS (inner product = cosine on normalized vectors)

In [5]:
import faiss

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
print("Stored:", index.ntotal, "sentences")

Stored: 4 sentences


## 5. Query

In [6]:
def search(query, k=1):
    q = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    return [(sentences[i], float(s)) for i, s in zip(ids[0], scores[0])]

query = input("Ask a question (e.g. Who does Alice know?): ")
for text, score in search(query):
    print(f"Answer: {text}  (similarity {score:.3f})")

Ask a question (e.g. Who does Alice know?): Who does Alice know?
Answer: Alice knows Bob  (similarity 0.844)
